<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/4%20-%20Redes%20Neurais/4.7_extra_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 4.7 — Autoencoder (extra)

Até aqui, todas as redes foram treinadas com aprendizado supervisionado: cada entrada tinha um rótulo (a espécie da flor, o dígito escrito) e a rede aprendia a prevê-lo. Neste notebook extra, a rede recebe uma imagem do MNIST e precisa devolver **a mesma imagem**. Não há rótulo: o alvo é a própria entrada.

Essa rede se chama autoencoder (autocodificador). Ela tem duas partes:

- um **codificador (encoder)**, que recebe a imagem e produz uma representação de dimensão baixa, a **codificação** (embedding);
- um **decodificador (decoder)**, que recebe a codificação e reconstrói a imagem.

Como a codificação tem muito menos números que a imagem (64 contra 784 pixels, neste notebook), a rede é obrigada a guardar só o essencial de cada dígito.

![Esquema de um autoencoder](https://raw.githubusercontent.com/vcasadei/images/master/autoencoder_schema.jpg)

*Esquema: imagem original (original input) → codificador (encoder) → representação comprimida (compressed representation) → decodificador (decoder) → imagem reconstruída (reconstructed input).*

## Objetivos

- Entender a estrutura de um autoencoder: codificador, codificação e decodificador.
- Treinar um autoencoder denso no MNIST usando a própria imagem como alvo, com `nn.MSELoss` e `optim.Adam`.
- Acompanhar a evolução das reconstruções ao longo das épocas.
- Explorar o espaço de codificação, interpolando entre as codificações de duas imagens.

## Importação das bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)  # reprodutibilidade

# O treino é leve (menos de 1 min em CPU), mas usa a GPU se houver.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Dados

Usamos o conjunto de treino do MNIST. `transforms.ToTensor()` converte cada imagem (28×28, valores inteiros de 0 a 255) num tensor `float` de shape `(1, 28, 28)` com valores em $[0, 1]$.

Para o treino rodar em CPU em menos de um minuto, usamos só as primeiras 4.096 imagens. `Subset(dataset, indices)` cria um `Dataset` com apenas esses índices, sem copiar os dados.

In [ ]:
mnist_full = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())

# Com GPU, dá para usar o conjunto inteiro: n_amostras = len(mnist_full)  (60.000 imagens)
n_amostras = 4096
mnist_data = Subset(mnist_full, range(n_amostras))
rotulos = mnist_full.targets[:n_amostras]   # dígitos das imagens usadas (só para escolher exemplos)

img, label = mnist_data[0]
print("imagens:", len(mnist_data))
print("shape de uma imagem:", tuple(img.shape), "| min:", img.min().item(), "| max:", img.max().item(), "| dígito:", label)

## Modelo

O autoencoder abaixo usa só camadas densas (`nn.Linear`), como as redes do módulo 4. Duas camadas novas ajustam o formato dos tensores, sem parâmetros:

- `nn.Flatten()`: achata cada imagem `(1, 28, 28)` num vetor de 784 pixels, mantendo a dimensão do mini-batch. Faz o mesmo que `x.view(-1, 784)`, mas como uma camada, que pode entrar num `nn.Sequential`;
- `nn.Unflatten(1, (1, 28, 28))`: o caminho inverso; transforma a dimensão 1 (os 784 valores) de volta em `(1, 28, 28)`.

Shapes de um mini-batch com $N$ imagens:

| etapa | camada | shape da saída |
|---|---|---|
| entrada | — | `(N, 1, 28, 28)` |
| codificador | `Flatten()` | `(N, 784)` |
| | `Linear(784, 256)` + ReLU | `(N, 256)` |
| | `Linear(256, 64)` | `(N, 64)` ← codificação: 64 números por imagem |
| decodificador | `Linear(64, 256)` + ReLU | `(N, 256)` |
| | `Linear(256, 784)` + sigmoide | `(N, 784)` |
| | `Unflatten(1, (1, 28, 28))` | `(N, 1, 28, 28)` |

`nn.Sequential(...)` encadeia camadas: a saída de uma é a entrada da seguinte. A codificação não tem ativação: os 64 números podem assumir qualquer valor real.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),                     # (N, 1, 28, 28) -> (N, 784)
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 64)                # codificação: 64 números por imagem
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28))      # (N, 784) -> (N, 1, 28, 28)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

A última ativação do decodificador é a sigmoide: os pixels das imagens estão em $[0, 1]$, e a sigmoide produz valores no mesmo intervalo.

Conferindo os shapes com um mini-batch de 8 imagens:

In [ ]:
x_teste = torch.stack([mnist_data[i][0] for i in range(8)])   # (8, 1, 28, 28)
modelo_teste = Autoencoder()
with torch.no_grad():                                           # só inspeção: não precisamos de gradientes
    codificacao = modelo_teste.encoder(x_teste)
    reconstrucao = modelo_teste.decoder(codificacao)
print("entrada:     ", tuple(x_teste.shape))
print("codificação: ", tuple(codificacao.shape))
print("reconstrução:", tuple(reconstrucao.shape))
print("parâmetros:  ", sum(p.numel() for p in modelo_teste.parameters()))

## Treinamento

Como treinar um autoencoder? Se passarmos uma imagem pelo codificador e o resultado pelo decodificador, queremos receber de volta, aproximadamente, a mesma imagem. Então minimizamos o **erro de reconstrução** nos dados de treino, com a função de custo (loss) `nn.MSELoss`, o erro quadrático médio calculado pixel a pixel:

$$J = \frac{1}{N \cdot 784} \sum_{n=1}^{N} \sum_{p=1}^{784} \left(\hat{x}_{n,p} - x_{n,p}\right)^2$$

em que $N$ é o tamanho do mini-batch, $x_{n,p}$ é o valor do pixel $p$ da imagem original $n$ e $\hat{x}_{n,p}$ é o mesmo pixel na reconstrução. Em código, `criterion(recon, img)`: o alvo é a própria entrada `img`, e o rótulo do dígito é descartado (`img, _`).

Fora a função de custo e o alvo, o laço de treino é o mesmo dos notebooks anteriores (`zero_grad` → `backward` → `step`). O otimizador é o Adam, comparado com outros otimizadores nos notebooks 4.4 e 4.5. `weight_decay=1e-5` acrescenta uma regularização L2 leve nos pesos (notebook 4.3).

A função `train` também guarda, ao fim de cada época, o último mini-batch de imagens e suas reconstruções, para vermos a evolução depois.

In [ ]:
def train(model, n_epochs=5, batch_size=64, lr=1e-3):
    torch.manual_seed(42)
    criterion = nn.MSELoss()  # erro quadrático médio por pixel
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    train_loader = DataLoader(mnist_data, batch_size=batch_size, shuffle=True)
    outputs = []   # (época, imagens, reconstruções) do último mini-batch de cada época
    losses = []    # custo médio de cada época
    model.train()
    for epoch in range(n_epochs):
        running_loss = 0.0
        for img, _ in train_loader:        # o rótulo do dígito não é usado
            img = img.to(device)
            recon = model(img)
            loss = criterion(recon, img)   # o alvo é a própria imagem
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * img.size(0)
        losses.append(running_loss / len(mnist_data))
        print(f"Época {epoch + 1:2d} | custo médio: {losses[-1]:.4f}")
        # .detach(): guarda só os valores, sem o histórico de gradientes; .cpu(): o matplotlib precisa de tensores na CPU
        outputs.append((epoch, img.detach().cpu(), recon.detach().cpu()))
    return outputs, losses

In [ ]:
torch.manual_seed(42)
model = Autoencoder().to(device)
n_epochs = 20   # alguns segundos em CPU com 4.096 imagens; com GPU e as 60.000 imagens, 20 épocas também são um bom valor
outputs, losses = train(model, n_epochs=n_epochs)

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(range(1, n_epochs + 1), losses, marker="o")
plt.xlabel("época")
plt.ylabel("custo médio (MSE)")
plt.title("Função de custo no treino")
plt.grid(True)
plt.show()

O custo cai ao longo do treino: as reconstruções ficam cada vez mais parecidas com as imagens originais.

Vamos ver a evolução. Em cada figura, a linha de cima mostra imagens do último mini-batch da época e a linha de baixo, as reconstruções correspondentes.

In [ ]:
def mostrar_pares(originais, reconstrucoes, titulo, n=9):
    plt.figure(figsize=(9, 2.2))
    plt.suptitle(titulo)
    for i in range(n):
        plt.subplot(2, n, i + 1)
        plt.imshow(originais[i][0], cmap="gray")
        plt.axis("off")
        plt.subplot(2, n, n + i + 1)
        plt.imshow(reconstrucoes[i][0], cmap="gray")
        plt.axis("off")
    plt.show()

for k in list(range(0, n_epochs, 5)) + [n_epochs - 1]:
    _, imgs, recon = outputs[k]
    mostrar_pares(imgs, recon, f"Época {k + 1}: originais (em cima) × reconstruções (embaixo)")

Nas primeiras épocas, as reconstruções não se parecem com as originais: lembram uma média de várias imagens de treino. Com o treino, ficam mais nítidas.

## Estrutura do espaço de codificação

Como a dimensão cai muito (784 → 64), a codificação precisa ter alguma estrutura: a rede economiza espaço levando imagens parecidas a codificações parecidas.

Vamos verificar isso. Primeiro, escolhemos duas imagens do **mesmo dígito** no conjunto de treino:

In [ ]:
idx = torch.where(rotulos == 2)[0][:2]       # as duas primeiras imagens do dígito 2
i1, i2 = idx[0].item(), idx[1].item()
x1, _ = mnist_data[i1]
x2, _ = mnist_data[i2]

plt.figure(figsize=(3, 1.8))
plt.subplot(1, 2, 1); plt.imshow(x1[0], cmap="gray"); plt.title(f"índice {i1}"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(x2[0], cmap="gray"); plt.title(f"índice {i2}"); plt.axis("off")
plt.show()

Calculamos as codificações das duas imagens aplicando só o codificador. `model.eval()` coloca o modelo em modo de avaliação e `torch.no_grad()` desliga o cálculo de gradientes, que não é necessário aqui:

In [ ]:
model.eval()
x = torch.stack([x1, x2]).to(device)    # (2, 1, 28, 28): juntamos as duas para chamar o codificador uma vez só
with torch.no_grad():
    embedding = model.encoder(x)        # (2, 64)
e1 = embedding[0]                       # codificação da primeira imagem
e2 = embedding[1]                       # codificação da segunda imagem
print("shape das codificações:", tuple(embedding.shape))

Agora vamos além de decodificar $e_1$ e $e_2$: vamos **interpolar** entre as duas codificações e decodificar os pontos intermediários. Para $\alpha$ variando de 0 a 1,

$$e(\alpha) = (1 - \alpha)\, e_1 + \alpha\, e_2$$

Com $\alpha = 0$ temos $e_1$; com $\alpha = 1$, $e_2$; os valores intermediários misturam as duas codificações.

In [ ]:
alphas = torch.linspace(0, 1, 10)                                        # 10 valores de 0 a 1
embedding_values = torch.stack([(1 - a) * e1 + a * e2 for a in alphas])  # (10, 64)
with torch.no_grad():
    recons = model.decoder(embedding_values).cpu()                       # (10, 1, 28, 28)
print(tuple(recons.shape))

Na linha de cima estão as reconstruções de cada codificação interpolada; embaixo, as imagens originais nas pontas (a primeira à esquerda, a segunda à direita):

In [ ]:
def mostrar_interpolacao(recons, x1, x2):
    n = len(recons)
    plt.figure(figsize=(10, 2.2))
    for i in range(n):
        plt.subplot(2, n, i + 1)
        plt.imshow(recons[i][0], cmap="gray")
        plt.axis("off")
    plt.subplot(2, n, n + 1); plt.imshow(x1[0], cmap="gray"); plt.axis("off")
    plt.subplot(2, n, 2 * n); plt.imshow(x2[0], cmap="gray"); plt.axis("off")
    plt.show()

mostrar_interpolacao(recons, x1, x2)

A transição entre as duas imagens é suave. As imagens do meio provavelmente são novas: nenhuma imagem de treino é exatamente igual a elas.

Podemos fazer o mesmo com dois dígitos **diferentes**. A função abaixo junta os passos anteriores; deve haver também uma transição suave entre os dois dígitos.

In [ ]:
def interpolate(index1, index2):
    x1, _ = mnist_data[index1]
    x2, _ = mnist_data[index2]
    model.eval()
    with torch.no_grad():
        embedding = model.encoder(torch.stack([x1, x2]).to(device))
        e1, e2 = embedding[0], embedding[1]
        alphas = torch.linspace(0, 1, 10)
        embedding_values = torch.stack([(1 - a) * e1 + a * e2 for a in alphas])
        recons = model.decoder(embedding_values).cpu()
    print(f"dígito {rotulos[index1].item()} → dígito {rotulos[index2].item()}")
    mostrar_interpolacao(recons, x1, x2)

interpolate(0, 1)

In [ ]:
interpolate(1, 10)

In [ ]:
interpolate(4, 5)

In [ ]:
interpolate(20, 30)

## Exercícios

**1.** Refaça o autoencoder com uma codificação de apenas **2** números e uma camada escondida a mais de cada lado, seguindo a arquitetura da figura abaixo: 784 → 512 → 128 → **2** → 128 → 512 → 784. Use ReLU entre as camadas, nenhuma ativação na codificação e sigmoide na saída.

Para reaproveitar a função `train` sem mudanças, o `forward` deve receber imagens `(N, 1, 28, 28)` e devolver o mesmo shape: use `x.view(-1, 784)` na entrada (o mesmo que `nn.Flatten()`; o `-1` deixa o PyTorch calcular $N$) e `.view(-1, 1, 28, 28)` na saída. Assim, `encoder` recebe vetores `(N, 784)`. As células seguintes do exercício usam esta classe.

![Arquitetura do autoencoder denso](https://raw.githubusercontent.com/vcasadei/images/master/autoencoder_1.png)

In [ ]:
class AutoencoderDenso(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: escreva sua solução aqui

    def forward(self, x):
        x = x.view(-1, 784)                  # (N, 1, 28, 28) -> (N, 784)
        # TODO: escreva sua solução aqui
        return x.view(-1, 1, 28, 28)         # volta ao formato de imagem

In [ ]:
torch.manual_seed(42)
model_denso = AutoencoderDenso().to(device)
print("parâmetros:", sum(p.numel() for p in model_denso.parameters()))
outputs_denso, losses_denso = train(model_denso, n_epochs=n_epochs)

_, imgs, recon = outputs_denso[-1]
mostrar_pares(imgs, recon, "Autoencoder denso, última época: originais × reconstruções")

**2.** Compare o custo final e as reconstruções do autoencoder com codificação de 2 números com os do autoencoder com codificação de 64. Qual reconstrói melhor? Compare também o número de parâmetros. O que explica o resultado?

*Escreva sua resposta aqui.*

**3.** Como a codificação do autoencoder denso tem só 2 números, ela pode ser desenhada num plano. Calcule a codificação das 4.096 imagens de treino com `model_denso.encoder` e faça um gráfico de dispersão, colorindo cada ponto pelo dígito (`rotulos`). Os dígitos formam grupos? Algum par de dígitos se mistura mais?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

## Aprendizados

- Um autoencoder tem codificador e decodificador e é treinado para reconstruir a própria entrada; não precisa de rótulos.
- O custo é o erro de reconstrução (`nn.MSELoss` pixel a pixel); a sigmoide na saída mantém os pixels em $[0, 1]$, o mesmo intervalo da entrada.
- O gargalo (codificação menor que a imagem) obriga a rede a guardar o essencial; quanto menor a codificação, piores as reconstruções.
- Imagens parecidas recebem codificações próximas: interpolar entre codificações gera transições suaves, e uma codificação 2D já separa boa parte dos dígitos.
- O laço de treino é o mesmo das redes supervisionadas; mudam apenas o alvo e a função de custo.